# Multi-Head 1D CNN Model

ECG Excel-file version for **normal / borderline / abnormal** data.

Updated: uses Google Drive paths, simple automatic head activation, named input dictionaries,
and explicit sample weights instead of `class_weight` for Colab/Keras compatibility.

This version is intentionally notebook-style and keeps the original cell-by-cell workflow.

Expected Google Drive / local folder structure:

```text
ECG_CNN_Project/
  data/
    normal.xlsx
    borderline.xlsx
    abnormal.xlsx
  outputs/
```

Each Excel file supplies one class. Each training sample is rebuilt from:

```text
File_Name + Section_Number
```

`Sample_In_Section` is used to determine the ideal segment length. Shorter vectors are expanded by repeating their values until they match the ideal length.

Each active signal column becomes one CNN input head. Missing signal columns automatically deactivate their heads.

## Google Drive Specific Commands

In Colab:

1. Put your files in a Google Drive folder like:

```text
MyDrive/ECG_CNN_Project/data/normal.xlsx
MyDrive/ECG_CNN_Project/data/borderline.xlsx
MyDrive/ECG_CNN_Project/data/abnormal.xlsx
```

2. Set `USE_GOOGLE_DRIVE = True`.
3. Edit `GOOGLE_DRIVE_PROJECT_FOLDER` if your folder name/path is different.

This version does **not** force TensorFlow onto a GPU. In Colab, use:

```text
Runtime > Change runtime type > T4 GPU
```

TensorFlow will use the GPU automatically when Colab exposes one.

In [ ]:
import os
import re
import sys
import pickle
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve,
    auc,
    f1_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    MaxPooling1D,
    Dropout,
    Flatten,
    Dense,
    concatenate,
)
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

print("TensorFlow version:", tf.__version__)

In [ ]:
# Drive/local path setup.
# Keep this cell simple: edit only these two lines in most cases.

USE_GOOGLE_DRIVE = True
GOOGLE_DRIVE_PROJECT_FOLDER = "/content/drive/MyDrive/ECG_CNN_Project"

# If running locally outside Colab, the code falls back to the current working directory.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = Path(GOOGLE_DRIVE_PROJECT_FOLDER)
elif USE_GOOGLE_DRIVE and not IN_COLAB:
    print("Google Drive mode was requested, but this does not appear to be Colab.")
    print("Falling back to the current working directory.")
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd()

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())

if DATA_ROOT.exists():
    print("Excel files in DATA_ROOT:")
    for p in sorted(DATA_ROOT.glob("*.xlsx")):
        print(" -", p.name)
else:
    print("DATA_ROOT does not exist yet. Make sure your Drive folder contains a data subfolder.")

## Simple GPU Check

This cell only reports whether TensorFlow sees a GPU. It does not force GPU placement.
That is usually safer in Colab and avoids device-placement errors.

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("Physical GPU devices detected by TensorFlow:", gpus)

if gpus:
    print("A GPU is visible. TensorFlow/Keras will normally use it automatically.")
else:
    print("No GPU is visible. In Colab, use Runtime > Change runtime type > GPU, then restart and run all.")

## Input Data Selection

In [ ]:
CLASS_NAMES = ["normal", "borderline", "abnormal"]

class_excel_files = {
    "normal": DATA_ROOT / "normal.xlsx",
    "borderline": DATA_ROOT / "borderline.xlsx",
    "abnormal": DATA_ROOT / "abnormal.xlsx",
}

SEGMENT_SHEET_NAME = "Cleaned_Segments"

for label, path in class_excel_files.items():
    print(f"{label:10s} -> {path} | exists: {path.exists()}")

## Data Import

Each Excel file is read as one class. The label is taken from the file name, not from a label column.

In [ ]:
ID_COLUMNS = ["File_Name", "Section_Number", "Sample_In_Section"]

NON_SIGNAL_COLUMNS = [
    "File_Name",
    "Section_Number",
    "Sample_In_Section",
    "Original_Index",
    "Time_s",
    "Class_Label",
    "Source_Excel_File",
]


def read_class_excel(label, excel_path, preferred_sheet=SEGMENT_SHEET_NAME):
    """Read one class Excel file and attach its class label."""
    excel_path = Path(excel_path)
    if not excel_path.exists():
        raise FileNotFoundError(f"Missing file for class '{label}': {excel_path}")

    with pd.ExcelFile(excel_path) as xls:
        if preferred_sheet in xls.sheet_names:
            sheet_name = preferred_sheet
        else:
            sheet_name = xls.sheet_names[0]
            print(f"Warning: {excel_path.name} does not contain sheet '{preferred_sheet}'. Using first sheet: {sheet_name}")

        df = pd.read_excel(xls, sheet_name=sheet_name)

    df.columns = [str(c).strip() for c in df.columns]

    missing = [c for c in ID_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"{excel_path.name} is missing required identifier columns: {missing}. "
            f"Available columns are: {df.columns.tolist()}"
        )

    df["Class_Label"] = label
    df["Source_Excel_File"] = excel_path.name
    return df

In [ ]:
all_dfs = {}

for label, path in class_excel_files.items():
    print(f"Reading {label}: {path}")
    df = read_class_excel(label, path)
    all_dfs[label] = df
    print(f"  shape: {df.shape}")
    print(f"  first columns: {df.columns.tolist()[:15]}")

combined_df = pd.concat(all_dfs.values(), ignore_index=True)
combined_df.columns = [str(c).strip() for c in combined_df.columns]

print("\nCombined dataframe shape:", combined_df.shape)
print("Class row counts:")
print(combined_df["Class_Label"].value_counts())
print("\nFirst rows:")
display(combined_df.head())

In [ ]:
pickle_file = OUTPUT_ROOT / "three_excel_class_dfs.pickle"
with open(pickle_file, "wb") as f:
    pickle.dump(all_dfs, f)

print("Saved:", pickle_file)

## Post Pickle Running Point

If the import has already been run, start here after the pickle has been made.

In [ ]:
with open(pickle_file, "rb") as f:
    loaded_dfs = pickle.load(f)

combined_df = pd.concat(loaded_dfs.values(), ignore_index=True)
combined_df.columns = [str(c).strip() for c in combined_df.columns]

print("Loaded classes:", list(loaded_dfs.keys()))
print("Combined dataframe shape:", combined_df.shape)

## Head Activation / Deactivation

This is the simplified head-control section.

Rules:

- Every numeric signal column becomes one CNN head.
- Identifier/debug columns are ignored.
- If a signal column is missing from one of the three Excel files, that head automatically turns off.
- If a signal column exists but has no numeric values in one class file, that head turns off.
- To manually turn off a head, put it in `HEADS_TO_DROP`.
- To use only a subset of heads, put them in `KEEP_ONLY_HEADS`.

In [ ]:
# Most ECG files use this order. The code will only activate columns that actually exist and contain data.
EXPECTED_HEAD_ORDER = [
    "I", "II", "III", "aVR", "aVL", "aVF",
    "V1", "V2", "V3", "V4", "V5", "V6",
]

# If you want the code to discover every possible numeric column instead, set this to None.
# EXPECTED_HEAD_ORDER = None

# Simple manual controls.
HEADS_TO_DROP = []
# Example:
# HEADS_TO_DROP = ["III"]

KEEP_ONLY_HEADS = None
# Example:
# KEEP_ONLY_HEADS = ["I", "II", "V1", "V2"]

MIN_NON_NULL_VALUES_PER_CLASS = 1


def numeric_valid_count(df, column_name):
    if column_name not in df.columns:
        return 0
    return int(pd.to_numeric(df[column_name], errors="coerce").notna().sum())


def discover_candidate_heads(loaded_dfs, expected_order=None):
    if expected_order is not None:
        return list(expected_order)

    candidates = []
    for df in loaded_dfs.values():
        for col in df.columns:
            col = str(col).strip()
            if col not in NON_SIGNAL_COLUMNS and col not in candidates:
                candidates.append(col)
    return candidates


candidate_heads = discover_candidate_heads(loaded_dfs, EXPECTED_HEAD_ORDER)

if KEEP_ONLY_HEADS is not None:
    candidate_heads = [h for h in candidate_heads if h in KEEP_ONLY_HEADS]

head_status_rows = []
active_signal_columns = []

for head in candidate_heads:
    reasons = []
    valid_counts = {}

    if head in NON_SIGNAL_COLUMNS:
        reasons.append("identifier/debug column")

    if head in HEADS_TO_DROP:
        reasons.append("listed in HEADS_TO_DROP")

    for label, df in loaded_dfs.items():
        df.columns = [str(c).strip() for c in df.columns]
        valid_count = numeric_valid_count(df, head)
        valid_counts[label] = valid_count

        if head not in df.columns:
            reasons.append(f"missing from {label}.xlsx")
        elif valid_count < MIN_NON_NULL_VALUES_PER_CLASS:
            reasons.append(f"no numeric data in {label}.xlsx")

    active = len(reasons) == 0

    if active:
        active_signal_columns.append(head)
        combined_df[head] = pd.to_numeric(combined_df[head], errors="coerce")

    head_status_rows.append({
        "Head": head,
        "Active": active,
        "Reason_If_Inactive": "; ".join(reasons),
        **{f"Valid_Values_{label}": valid_counts.get(label, 0) for label in loaded_dfs.keys()},
    })

head_status_df = pd.DataFrame(head_status_rows)

print("Head status:")
display(head_status_df)

print("\nActive heads:", active_signal_columns)
print("Number of active heads:", len(active_signal_columns))

if len(active_signal_columns) == 0:
    raise ValueError("No active heads were detected. Check your Excel signal columns and identifier columns.")

# Original notebook variable style.
sheet_names = active_signal_columns
print("sheet_names / active heads:", sheet_names)

## Simple Per-Head CNN Settings

This is the simplified way to make unique CNNs for individual inputs.

Format for each block:

```python
(filters, kernel_size, pool_size, dropout)
```

Any head not listed in `CUSTOM_CNN_BLOCKS` uses `DEFAULT_CNN_BLOCKS`.

In [ ]:
DEFAULT_CNN_BLOCKS = [
    (64, 9, 2, 0.20),
    (128, 5, 2, 0.25),
    (256, 3, None, 0.30),
]

CUSTOM_CNN_BLOCKS = {
    # Example: give V1 a different CNN than the other leads.
    # "V1": [
    #     (32, 11, 2, 0.15),
    #     (64, 7, 2, 0.20),
    #     (128, 3, None, 0.25),
    # ],

    # Example: give Lead I a smaller CNN.
    # "I": [
    #     (32, 9, 2, 0.15),
    #     (64, 5, None, 0.20),
    # ],
}

print("Default CNN blocks:", DEFAULT_CNN_BLOCKS)
print("Custom CNN heads:", list(CUSTOM_CNN_BLOCKS.keys()))

## Load and Filter Sheet

This version does not use separate Excel sheets as heads. The active signal columns are the heads.

In [ ]:
optional_debug_columns = [c for c in ["Original_Index", "Time_s"] if c in combined_df.columns]
needed_columns = ID_COLUMNS + optional_debug_columns + active_signal_columns + ["Class_Label", "Source_Excel_File"]

filtered_df = combined_df[needed_columns].copy()

filtered_df["File_Name"] = filtered_df["File_Name"].astype(str)
filtered_df["Section_Number"] = pd.to_numeric(filtered_df["Section_Number"], errors="coerce")
filtered_df["Sample_In_Section"] = pd.to_numeric(filtered_df["Sample_In_Section"], errors="coerce")

before_drop = len(filtered_df)
filtered_df = filtered_df.dropna(subset=["File_Name", "Section_Number", "Sample_In_Section"]).copy()
after_drop = len(filtered_df)

filtered_df["Section_Number"] = filtered_df["Section_Number"].astype(int)
filtered_df["Sample_In_Section"] = filtered_df["Sample_In_Section"].astype(int)

print("Filtered dataframe shape:", filtered_df.shape)
print("Dropped rows with missing identifiers:", before_drop - after_drop)
display(filtered_df.head())

## Train and Validation Data Isolation

Split by `File_Name`, not by row, so sections from the same subject do not leak across train/validation/test.

In [ ]:
USE_MANUAL_SPLIT = False

test_folders = []
validation_folders = []

# Example:
# test_folders = ["Subject_001", "Subject_002"]
# validation_folders = ["Subject_003", "Subject_004"]

print("USE_MANUAL_SPLIT:", USE_MANUAL_SPLIT)
print("Manual test_folders:", test_folders)
print("Manual validation_folders:", validation_folders)

## Randomization Algorithm

This chooses held-out `File_Name` values per class.

In [ ]:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

TEST_SUBJECTS_PER_CLASS = 2
VALIDATION_SUBJECTS_PER_CLASS = 2

USE_FIXED_SUBJECT_COUNTS_PER_CLASS = True
TEST_FRACTION_PER_CLASS = 0.15
VALIDATION_FRACTION_PER_CLASS = 0.15


def choose_split_subjects_by_class(sample_meta_df):
    selected_test = []
    selected_validation = []

    for label in CLASS_NAMES:
        subjects = sorted(sample_meta_df.loc[sample_meta_df["Class_Label"] == label, "Parent Folder"].unique())
        subjects = list(subjects)
        random.shuffle(subjects)

        n_subjects = len(subjects)
        if n_subjects == 0:
            print(f"Warning: class '{label}' has no subjects.")
            continue

        if USE_FIXED_SUBJECT_COUNTS_PER_CLASS:
            n_test = min(TEST_SUBJECTS_PER_CLASS, max(0, n_subjects - 1))
            n_valid = min(VALIDATION_SUBJECTS_PER_CLASS, max(0, n_subjects - n_test - 1))
        else:
            n_test = max(1, int(round(n_subjects * TEST_FRACTION_PER_CLASS))) if n_subjects >= 3 else 0
            n_valid = max(1, int(round(n_subjects * VALIDATION_FRACTION_PER_CLASS))) if n_subjects >= 4 else 0
            if n_test + n_valid >= n_subjects:
                n_valid = max(0, n_subjects - n_test - 1)

        class_test = subjects[:n_test]
        class_validation = subjects[n_test:n_test + n_valid]

        selected_test.extend(class_test)
        selected_validation.extend(class_validation)

        print(
            f"{label}: subjects={n_subjects}, "
            f"test={len(class_test)}, validation={len(class_validation)}, "
            f"train={n_subjects-len(class_test)-len(class_validation)}"
        )

    return selected_test, selected_validation

## Data Preparations

Each sample is one `File_Name + Section_Number` segment.

In [ ]:
sample_counter = pd.to_numeric(filtered_df["Sample_In_Section"], errors="coerce")
global_min_sample_counter = int(sample_counter.min())
global_max_sample_counter = int(sample_counter.max())

sample_counter_is_zero_based = global_min_sample_counter == 0
length_from_counter = global_max_sample_counter + 1 if sample_counter_is_zero_based else global_max_sample_counter

section_length_df = (
    filtered_df
    .groupby(["Class_Label", "File_Name", "Section_Number"], dropna=False)
    .agg(
        Row_Count=("Sample_In_Section", "size"),
        Min_Sample_In_Section=("Sample_In_Section", "min"),
        Max_Sample_In_Section=("Sample_In_Section", "max"),
    )
    .reset_index()
)

max_row_count = int(section_length_df["Row_Count"].max())
IDEAL_LENGTH = int(max(length_from_counter, max_row_count))

print("Sample_In_Section min:", global_min_sample_counter)
print("Sample_In_Section max:", global_max_sample_counter)
print("Sample_In_Section appears zero-based:", sample_counter_is_zero_based)
print("Length from Sample_In_Section:", length_from_counter)
print("Maximum row count in any File_Name + Section_Number group:", max_row_count)
print("IDEAL_LENGTH used for CNN:", IDEAL_LENGTH)

print("\nSection length summary:")
display(section_length_df[["Row_Count", "Max_Sample_In_Section"]].describe())

In [ ]:
PADDING_MODE = "repeat_vector"  # options: "repeat_vector", "repeat_last", "zero"


def normalize_segment_length(values_2d, target_length, mode=PADDING_MODE):
    """
    values_2d shape: timesteps x channels
    Returns shape: target_length x channels
    """
    values_2d = np.asarray(values_2d, dtype=np.float32)
    current_length = values_2d.shape[0]

    if current_length == 0:
        raise ValueError("Cannot normalize an empty segment.")

    if current_length == target_length:
        return values_2d

    if current_length > target_length:
        return values_2d[:target_length, :]

    pad_length = target_length - current_length

    if mode == "repeat_vector":
        repeats = int(np.ceil(target_length / current_length))
        expanded = np.tile(values_2d, (repeats, 1))
        return expanded[:target_length, :]

    if mode == "repeat_last":
        last_row = values_2d[-1:, :]
        padding = np.repeat(last_row, pad_length, axis=0)
        return np.vstack([values_2d, padding])

    if mode == "zero":
        padding = np.zeros((pad_length, values_2d.shape[1]), dtype=np.float32)
        return np.vstack([values_2d, padding])

    raise ValueError(f"Unknown PADDING_MODE: {mode}")

In [ ]:
X_list = []
y = []
meta_rows = []

sort_columns = ["Class_Label", "File_Name", "Section_Number", "Sample_In_Section"]
filtered_df = filtered_df.sort_values(sort_columns).reset_index(drop=True)

for (class_label, file_name, section_number), group in filtered_df.groupby(
    ["Class_Label", "File_Name", "Section_Number"], sort=False
):
    group = group.sort_values("Sample_In_Section")

    segment_values = group[active_signal_columns].copy()
    segment_values = segment_values.apply(pd.to_numeric, errors="coerce")

    segment_values = (
        segment_values
        .interpolate(axis=0, limit_direction="both")
        .ffill()
        .bfill()
        .fillna(0.0)
    )

    segment_array = segment_values.to_numpy(dtype=np.float32)
    original_length = segment_array.shape[0]
    normalized_array = normalize_segment_length(segment_array, IDEAL_LENGTH, mode=PADDING_MODE)

    X_list.append(normalized_array)
    y.append(class_label)

    meta_rows.append({
        "File_Name": file_name,
        "Section_Number": int(section_number),
        "Parent Folder": file_name,
        "Class_Label": class_label,
        "Original_Length": original_length,
        "Normalized_Length": normalized_array.shape[0],
        "Min_Sample_In_Section": int(group["Sample_In_Section"].min()),
        "Max_Sample_In_Section": int(group["Sample_In_Section"].max()),
        "Source_Excel_File": group["Source_Excel_File"].iloc[0],
    })

X = np.stack(X_list, axis=0).astype(np.float32)
y = np.array(y)
sample_meta_df = pd.DataFrame(meta_rows)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Number of samples/segments:", len(sample_meta_df))
print("Number of active heads:", len(active_signal_columns))
print("IDEAL_LENGTH:", IDEAL_LENGTH)

print("\nSample metadata:")
display(sample_meta_df.head())

## Data Distribution Check

In [ ]:
y_labels = y.ravel()
classes = len(np.unique(y_labels))
print("Number of unique classes:", classes)
print("Unique values:", np.unique(y_labels))

print("\nSegment/sample counts by class:")
print(pd.Series(y_labels).value_counts())

print("\nSubject/File_Name counts by class:")
display(sample_meta_df.groupby("Class_Label")["Parent Folder"].nunique().rename("Unique_File_Name_Count"))

print("\nSection counts by class:")
display(sample_meta_df.groupby("Class_Label")["Section_Number"].count().rename("Segment_Count"))

## Train-Test-Validation Data Isolation

In [ ]:
if not USE_MANUAL_SPLIT:
    test_folders, validation_folders = choose_split_subjects_by_class(sample_meta_df)

print("\nFinal test_folders:")
print(test_folders)
print("\nFinal validation_folders:")
print(validation_folders)

train_indices = sample_meta_df[
    ~(sample_meta_df["Parent Folder"].isin(test_folders) | sample_meta_df["Parent Folder"].isin(validation_folders))
].index.values

validation_indices = sample_meta_df[sample_meta_df["Parent Folder"].isin(validation_folders)].index.values

test_indices = sample_meta_df[sample_meta_df["Parent Folder"].isin(test_folders)].index.values

print("\nTrain Indices:", train_indices)
print("Validation Indices:", validation_indices)
print("Test Indices:", test_indices)

print("\nNumber of train samples:", len(train_indices))
print("Number of validation samples:", len(validation_indices))
print("Number of test samples:", len(test_indices))

## Combining Data for Multi-Head CNN Model

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Number of heads/channels:", X.shape[2])
print("Active signal columns:", active_signal_columns)

## Data Leakage Control Algorithm

In [ ]:
def check_overlap(train_indices, validation_indices, test_indices):
    train_set = set(train_indices)
    validation_set = set(validation_indices)
    test_set = set(test_indices)

    if train_set.intersection(validation_set):
        raise ValueError("Overlap found between train and validation indices.")
    if train_set.intersection(test_set):
        raise ValueError("Overlap found between train and test indices.")
    if validation_set.intersection(test_set):
        raise ValueError("Overlap found between validation and test indices.")

    print("No index overlap found among train, validation, and test sets.")


def check_subject_leakage(sample_meta_df, train_indices, validation_indices, test_indices):
    train_subjects = set(sample_meta_df.loc[train_indices, "Parent Folder"])
    validation_subjects = set(sample_meta_df.loc[validation_indices, "Parent Folder"])
    test_subjects = set(sample_meta_df.loc[test_indices, "Parent Folder"])

    if train_subjects.intersection(validation_subjects):
        raise ValueError(f"Subject leakage between train and validation: {train_subjects.intersection(validation_subjects)}")
    if train_subjects.intersection(test_subjects):
        raise ValueError(f"Subject leakage between train and test: {train_subjects.intersection(test_subjects)}")
    if validation_subjects.intersection(test_subjects):
        raise ValueError(f"Subject leakage between validation and test: {validation_subjects.intersection(test_subjects)}")

    print("No File_Name / Parent Folder leakage found among train, validation, and test sets.")


check_overlap(train_indices, validation_indices, test_indices)
check_subject_leakage(sample_meta_df, train_indices, validation_indices, test_indices)
print("Overlap check passed")

## Data Splitting

In [ ]:
X_train, y_train = X[train_indices], y[train_indices]
X_valid, y_valid = X[validation_indices], y[validation_indices]
X_test, y_test = X[test_indices], y[test_indices]

train_meta_df = sample_meta_df.iloc[train_indices].reset_index(drop=True)
valid_meta_df = sample_meta_df.iloc[validation_indices].reset_index(drop=True)
test_meta_df = sample_meta_df.iloc[test_indices].reset_index(drop=True)

print("Train set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_valid.shape, y_valid.shape)
print("Test set shape:", X_test.shape, y_test.shape)

print("\nTrain labels:")
print(pd.Series(y_train).value_counts())
print("\nValidation labels:")
print(pd.Series(y_valid).value_counts())
print("\nTest labels:")
print(pd.Series(y_test).value_counts())

if len(X_train) == 0:
    raise ValueError("No training samples selected. Reduce TEST_SUBJECTS_PER_CLASS / VALIDATION_SUBJECTS_PER_CLASS or use manual split.")
if len(X_valid) == 0:
    print("Warning: no validation samples selected. Model training with validation_data will fail unless you adjust the split.")
if len(X_test) == 0:
    print("Warning: no test samples selected. Evaluation will fail unless you adjust the split.")

## Signal Normalization

Standardize using only the training set.

In [ ]:
channel_mean = X_train.mean(axis=(0, 1), keepdims=True)
channel_std = X_train.std(axis=(0, 1), keepdims=True)
channel_std = np.where(channel_std == 0, 1.0, channel_std)

X_train = (X_train - channel_mean) / channel_std
X_valid = (X_valid - channel_mean) / channel_std
X_test = (X_test - channel_mean) / channel_std

print("Training data mean after standardization:", X_train.mean(axis=(0, 1)))
print("Training data std after standardization:", X_train.std(axis=(0, 1)))

## Class Balancer

In [ ]:
APPLY_CLASS_BALANCER = False


def balance_classes(X_data, y_data, random_seed=RANDOM_SEED):
    rng = np.random.default_rng(random_seed)
    unique_classes, counts = np.unique(y_data, return_counts=True)
    min_count = min(counts)

    X_balanced = []
    y_balanced = []

    for cls in unique_classes:
        cls_indices = np.where(y_data == cls)[0]
        rng.shuffle(cls_indices)
        selected_indices = cls_indices[:min_count]
        X_balanced.append(X_data[selected_indices])
        y_balanced.append(y_data[selected_indices])

    X_balanced = np.concatenate(X_balanced, axis=0)
    y_balanced = np.concatenate(y_balanced, axis=0)

    shuffle_indices = rng.permutation(len(y_balanced))
    return X_balanced[shuffle_indices], y_balanced[shuffle_indices]


def print_class_distribution(y_data, dataset_name):
    unique_classes, counts = np.unique(y_data, return_counts=True)
    print(f"{dataset_name} class distribution:")
    for cls, count in zip(unique_classes, counts):
        print(f"Class {cls}: {count} samples")
    print()


if APPLY_CLASS_BALANCER:
    X_train, y_train = balance_classes(X_train, y_train)
    print("Balanced Train set shape:", X_train.shape, y_train.shape)
else:
    print("Class balancer is OFF. Using class weights during training instead.")

print_class_distribution(y_train, "Train")
print_class_distribution(y_valid, "Validation")
print_class_distribution(y_test, "Test")

## Label Encoding

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(CLASS_NAMES)

y_train_encoded = label_encoder.transform(y_train)
y_valid_encoded = label_encoder.transform(y_valid)
y_test_encoded = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

print("Label mapping:")
for idx, class_name in enumerate(label_encoder.classes_):
    print(idx, "->", class_name)

In [ ]:
y_train_one_hot = to_categorical(y_train_encoded, num_classes=num_classes)
y_valid_one_hot = to_categorical(y_valid_encoded, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test_encoded, num_classes=num_classes)

print("y_train_one_hot shape:", y_train_one_hot.shape)
print("y_valid_one_hot shape:", y_valid_one_hot.shape)
print("y_test_one_hot shape:", y_test_one_hot.shape)

## Multi-Head Input Preparation

Keras/Colab is more reliable with multi-input models when the inputs are provided
as a dictionary keyed by the actual model input-layer names.

This avoids obscure graph errors that can occur when using a long list of inputs
together with weighting arguments.

In [ ]:
def clean_layer_name(name):
    name = re.sub(r"[^A-Za-z0-9_]+", "_", str(name))
    name = re.sub(r"_+", "_", name).strip("_")
    return name if name else "signal"


def input_name_for_head(head_name):
    return f"{clean_layer_name(head_name)}_input"


def prepare_multi_head_inputs(X_array, active_heads):
    """
    Convert samples x timesteps x channels into a dictionary of channel-specific inputs.

    Returns:
        {input_layer_name: array with shape samples x timesteps x 1}
    """
    if X_array.shape[2] != len(active_heads):
        raise ValueError(
            f"X has {X_array.shape[2]} channels, but active_heads has {len(active_heads)} heads."
        )

    input_dict = {}
    for channel_index, head_name in enumerate(active_heads):
        input_dict[input_name_for_head(head_name)] = X_array[:, :, channel_index:channel_index+1].astype(np.float32)

    return input_dict


X_train_inputs = prepare_multi_head_inputs(X_train, active_signal_columns)
X_valid_inputs = prepare_multi_head_inputs(X_valid, active_signal_columns)
X_test_inputs = prepare_multi_head_inputs(X_test, active_signal_columns)

print("Number of model inputs:", len(X_train_inputs))
print("Input dictionary keys:")
print(list(X_train_inputs.keys()))
print("Shape of first input:", next(iter(X_train_inputs.values())).shape)

print("\nInput shape check:")
for input_name, arr in X_train_inputs.items():
    print(f"{input_name:15s} {arr.shape}")

## Multi-Head CNN Architecture Definition

Each active head gets its own CNN branch. The branch is chosen by:

```python
blocks = CUSTOM_CNN_BLOCKS.get(head_name, DEFAULT_CNN_BLOCKS)
```

That is the only head-architecture logic.

In [ ]:
def build_head_branch(channel_input, head_name, blocks):
    safe_name = clean_layer_name(head_name)
    x = channel_input

    for block_idx, (filters, kernel_size, pool_size, dropout_rate) in enumerate(blocks, start=1):
        x = Conv1D(
            filters=int(filters),
            kernel_size=int(kernel_size),
            padding="same",
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001),
            name=f"{safe_name}_conv{block_idx}",
        )(x)
        x = BatchNormalization(name=f"{safe_name}_bn{block_idx}")(x)

        if pool_size is not None:
            x = MaxPooling1D(pool_size=int(pool_size), padding="same", name=f"{safe_name}_pool{block_idx}")(x)

        if dropout_rate is not None and float(dropout_rate) > 0:
            x = Dropout(float(dropout_rate), name=f"{safe_name}_drop{block_idx}")(x)

    x = Flatten(name=f"{safe_name}_flatten")(x)
    return x


tf.keras.backend.clear_session()

input_length = X_train.shape[1]
channels = []
convolutions = []
architecture_rows = []

for head_name in active_signal_columns:
    safe_name = clean_layer_name(head_name)
    blocks = CUSTOM_CNN_BLOCKS.get(head_name, DEFAULT_CNN_BLOCKS)

    channel_input = Input(shape=(input_length, 1), name=input_name_for_head(head_name))
    head_output = build_head_branch(channel_input, head_name, blocks)

    channels.append(channel_input)
    convolutions.append(head_output)
    architecture_rows.append({
        "Head": head_name,
        "Uses_Custom_CNN": head_name in CUSTOM_CNN_BLOCKS,
        "Blocks": blocks,
    })

if len(convolutions) > 1:
    merged = concatenate(convolutions, name="merged_heads")
else:
    merged = convolutions[0]

dense1 = Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001), name="dense_1")(merged)
dense1 = Dropout(0.40, name="dense_1_dropout")(dense1)

dense2 = Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001), name="dense_2")(dense1)
dense2 = Dropout(0.30, name="dense_2_dropout")(dense2)

output = Dense(num_classes, activation="softmax", name="class_output")(dense2)

model = Model(inputs=channels, outputs=output)
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print("Model created with", len(channels), "active heads.")
print("TensorFlow GPU devices visible at model build:", tf.config.list_physical_devices("GPU"))
display(pd.DataFrame(architecture_rows))

In [ ]:
model.summary()

## Define Early Stopping Conditions, One-Hot Encode, Compile and Run Model

In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

classes_present = np.unique(y_train_encoded)
class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=classes_present,
    y=y_train_encoded,
)
class_weights = {int(cls): float(weight) for cls, weight in zip(classes_present, class_weight_values)}
print("Class weights:", class_weights)

# IMPORTANT:
# In newer Colab/Keras versions, class_weight can throw obscure graph errors
# with multi-input models. We use explicit per-sample weights instead.
USE_SAMPLE_WEIGHTS = True

if USE_SAMPLE_WEIGHTS:
    train_sample_weights = np.array(
        [class_weights[int(class_id)] for class_id in y_train_encoded],
        dtype=np.float32,
    )
    print("Using explicit train_sample_weights instead of class_weight.")
    print("train_sample_weights shape:", train_sample_weights.shape)
    print("train_sample_weights first 10:", train_sample_weights[:10])
else:
    train_sample_weights = None
    print("Sample weighting is OFF.")

model_checkpoint_path = OUTPUT_ROOT / "best_multi_head_ecg_model.keras"

callbacks = [
    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6),
    ModelCheckpoint(str(model_checkpoint_path), monitor="val_loss", save_best_only=True),
]

history = model.fit(
    X_train_inputs,
    y_train_one_hot,
    validation_data=(X_valid_inputs, y_valid_one_hot),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    sample_weight=train_sample_weights,
    verbose=1,
)

## Training Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history.get("loss", []), label="Training Loss")
plt.plot(history.history.get("val_loss", []), label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history.get("accuracy", []), label="Training Accuracy")
plt.plot(history.history.get("val_accuracy", []), label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Testing

In [ ]:
print("Number of test samples being evaluated:", X_test.shape[0])
print("Test class distribution:")
print(pd.Series(y_test).value_counts())

if X_test.shape[0] == 0:
    raise ValueError("No test samples were selected. Increase dataset size or adjust TEST_SUBJECTS_PER_CLASS / manual test_folders.")

test_loss, test_accuracy = model.evaluate(X_test_inputs, y_test_one_hot, verbose=0)
y_test_prob = model.predict(X_test_inputs)

print("Test loss:", test_loss)
print("Test accuracy:", test_accuracy)

y_test_pred_encoded = np.argmax(y_test_prob, axis=1)
y_test_pred = label_encoder.inverse_transform(y_test_pred_encoded)

print("\nClassification report:")
print(classification_report(y_test, y_test_pred, labels=label_encoder.classes_, zero_division=0))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=label_encoder.classes_)

plt.figure(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.show()

## ROC/AUC

In [ ]:
plt.figure(figsize=(8, 6))

for class_index, class_name in enumerate(label_encoder.classes_):
    y_true_binary = y_test_one_hot[:, class_index]

    if len(np.unique(y_true_binary)) < 2:
        print(f"Skipping ROC for {class_name}: test set does not contain both positive and negative examples.")
        continue

    fpr, tpr, _ = roc_curve(y_true_binary, y_test_prob[:, class_index])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} AUC = {roc_auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("One-vs-Rest ROC Curves")
plt.legend()
plt.grid(True)
plt.show()

## F1 Score

In [ ]:
macro_f1 = f1_score(y_test, y_test_pred, labels=label_encoder.classes_, average="macro", zero_division=0)
weighted_f1 = f1_score(y_test, y_test_pred, labels=label_encoder.classes_, average="weighted", zero_division=0)

print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

## Save Outputs

In [ ]:
results_pickle_file = OUTPUT_ROOT / "three_excel_multi_head_cnn_results.pickle"

results = {
    "active_signal_columns": active_signal_columns,
    "head_status_df": head_status_df,
    "architecture_summary": pd.DataFrame(architecture_rows),
    "IDEAL_LENGTH": IDEAL_LENGTH,
    "PADDING_MODE": PADDING_MODE,
    "label_classes": label_encoder.classes_.tolist(),
    "test_folders": test_folders,
    "validation_folders": validation_folders,
    "sample_meta_df": sample_meta_df,
    "train_meta_df": train_meta_df,
    "valid_meta_df": valid_meta_df,
    "test_meta_df": test_meta_df,
    "history": history.history,
    "y_test": y_test,
    "y_test_pred": y_test_pred,
    "y_test_prob": y_test_prob,
}

with open(results_pickle_file, "wb") as f:
    pickle.dump(results, f)

print("Saved results to:", results_pickle_file)
print("Saved best model to:", model_checkpoint_path)